# Week 3 – Data Contract

**Name:** Muhammad Usman Shakir

**Repository:** flyrank-ml-internship

**Phase:** Foundations

This notebook defines the data contract for the Content Refresh ML lane, verifies the warehouse data, creates a feature frame, demonstrates feature leakage, and documents limitations.

In [17]:
%pip -q install duckdb huggingface_hub pandas

In [18]:
import duckdb
import pandas as pd
from huggingface_hub import login
from google.colab import userdata

In [19]:
HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face login successful!")

Hugging Face login successful!


In [20]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    print(f"Creating {name}...")
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM {src}")

print("✅ Views created successfully!")

Creating dim_clients...
Creating dim_content...
Creating fact_daily...
Creating fact_daily_sample...
Creating fact_query_90d...
✅ Views created successfully!


In [21]:
con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_daily
3,fact_daily_sample
4,fact_query_90d


# 1. Data Contract

### Q1. What does one row represent?

One row represents the daily performance metrics for one content page on one reporting date.

---

### Q2. Which table(s) will you use?

Primary table:

- fact_daily

Supporting table:

- dim_content

---

### Q3. Which time window will you use?

March 2026 (month = '2026-03')

---

### Q4. What will you predict?

This is a Ranking / Scoring machine learning problem.

The objective is to identify content pages most likely to benefit from a content refresh.

---

### Q5. What do you deliberately exclude?

Future information, label-derived variables and any information unavailable at prediction time.

# 2. Verification Queries

In [22]:
query1 = """
SELECT
    content_hash_id,
    report_date,
    COUNT(*) AS records
FROM fact_daily
WHERE month = '2026-03'
GROUP BY content_hash_id, report_date
ORDER BY records DESC
LIMIT 10;
"""

con.sql(query1).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,records
0,content_b7e512995f79d5a6,2026-03-01,1
1,content_a7da352b73b02668,2026-03-01,1
2,content_d056587ff7faca0c,2026-03-01,1
3,content_bfd1e41c2af250c8,2026-03-01,1
4,content_2662845f598544ef,2026-03-01,1
5,content_f39be42b42a4e8f6,2026-03-01,1
6,content_1855a661b4d36130,2026-03-01,1
7,content_22c063002b7c1caf,2026-03-01,1
8,content_0ea64f25303c9a77,2026-03-01,1
9,content_d720dde3701523c0,2026-03-01,1


### Query 1 Result

This query verifies the grain of the dataset. Each row represents one content page (`content_hash_id`) on one reporting date (`report_date`). Since every combination appears exactly once, the expected grain is confirmed.

In [23]:
query2 = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_daily
WHERE month = '2026-03';
"""

con.sql(query2).df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Query 2 Result

This query confirms the number of records available for March 2026 and verifies the beginning and ending dates of the selected analysis period.

In [24]:
query3 = """
SELECT
    COUNT(*) AS available_rows
FROM fact_daily
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE;
"""

con.sql(query3).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,364347


### Query 3 Result

This query verifies that only rows with both Google Search Console (GSC) and Google Analytics 4 (GA4) data available are included in the analysis.

# 3. Feature Frame

The following features are available at the decision time and can be used by a machine learning model to rank content pages for refresh.

In [25]:
features = """
SELECT
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_pageviews
FROM fact_daily
WHERE month='2026-03'
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE
LIMIT 10;
"""

feature_df = con.sql(features).df()
feature_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_pageviews
0,content_5c80451459c29b4a,2026-03-01,5,0,5.400000,1,1
1,content_b1f61fc81b28b2d4,2026-03-01,39,0,5.666667,2,2
2,content_e25ea7297a1dffd3,2026-03-01,179,0,5.156425,2,2
3,content_6b0149a80607dac3,2026-03-01,72,0,7.694444,1,1
4,content_62673eea26c31c17,2026-03-01,3282,1,6.167885,1,1
5,content_872342e050545a12,2026-03-01,39,0,6.538462,1,1
6,content_3c286ded8bd68120,2026-03-01,88,1,8.431818,1,1
7,content_b2108e8fe3360fa6,2026-03-01,40,1,5.300000,1,1
8,content_4c185d1c173cd53d,2026-03-01,23,0,30.304348,1,2
9,content_bd07be40ea0d5f54,2026-03-01,23,0,5.478261,1,1


### Feature Availability

**gsc_impressions** – Available when the decision is made because impressions are historical.

**gsc_clicks** – Historical search clicks already collected.

**gsc_avg_position** – Historical average ranking position.

**ga4_sessions** – Previous user sessions available before prediction.

**ga4_pageviews** – Historical pageviews collected before prediction.

In [26]:
leak_query = """
SELECT
    gsc_impressions,
    gsc_clicks,
    ga4_sessions,
    ga4_pageviews,
    gsc_clicks AS leaked_label
FROM fact_daily
WHERE month='2026-03'
LIMIT 10;
"""

con.sql(leak_query).df()

,gsc_impressions,gsc_clicks,ga4_sessions,ga4_pageviews,leaked_label
0,20,0,<NA>,<NA>,0
1,1,0,<NA>,<NA>,0
2,125,1,<NA>,<NA>,1
3,7,0,<NA>,<NA>,0
4,11,0,<NA>,<NA>,0
5,239,1,<NA>,<NA>,1
6,191,0,<NA>,<NA>,0
7,55,0,<NA>,<NA>,0
8,77,0,<NA>,<NA>,0
9,2,0,<NA>,<NA>,0


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [28]:
model_df = con.sql("""
SELECT
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_pageviews,
    gsc_clicks
FROM fact_daily
WHERE month='2026-03'
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE
LIMIT 5000
""").df()

model_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_pageviews,gsc_clicks
0,5,5.400000,1,1,0
1,39,5.666667,2,2,0
2,179,5.156425,2,2,0
3,72,7.694444,1,1,0
4,3282,6.167885,1,1,1


In [29]:
X = model_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_pageviews"
    ]
]

y = model_df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest R² Score:", r2_score(y_test, pred))

Honest R² Score: 0.5410424775416451


### Honest Model

The model only uses information available before the prediction.

This represents a realistic machine learning workflow.

In [30]:
X_leak = model_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_pageviews",
        "gsc_clicks"
    ]
]

y = model_df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

leak_model = RandomForestRegressor(
    random_state=42
)

leak_model.fit(X_train, y_train)

pred = leak_model.predict(X_test)

print("Leaked R² Score:", r2_score(y_test, pred))

Leaked R² Score: 0.9147494791389974


### Leakage Demonstration

The feature **gsc_clicks** is also the prediction target.

Including it as an input gives the model access to the answer, producing an unrealistically high score.

This is called **data leakage** and must always be avoided in production ML systems.

# Feature Leakage

The column **gsc_clicks** is intentionally reused as a label to demonstrate leakage.

A model trained with this information would achieve unrealistically high performance because it already contains the answer.

The leaked feature must therefore be removed before model training.

# 5. Limitation

This notebook analyses only one month (March 2026). Longer historical periods may provide more robust feature engineering and better generalization.

# 6. Self Check

✔ Defined the data contract.

✔ Verified the data grain.

✔ Verified the row count and reporting period.

✔ Verified data availability using `IS TRUE`.

✔ Built a five-feature frame.

✔ Demonstrated feature leakage.

✔ Documented one limitation.

## Conclusion

This notebook completed the required Week 3 tasks:

- Defined the data contract.
- Verified the dataset grain.
- Verified the reporting period.
- Checked data availability using `IS TRUE`.
- Built a five-feature dataset.
- Demonstrated feature leakage.
- Compared an honest model with a leaked model.
- Documented one limitation.

This exercise highlights why defining a correct data contract and preventing leakage are essential steps before training machine learning models.